###Using SQL To Work With Delta Table.

In [0]:
%sql
CREATE OR REPLACE TABLE delta_catalog.delta_db.invoices_ttv AS --(CTAS)
SELECT
  *
FROM
  PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_1_100.parquet`;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv;

#### Performing Delete,Update,Write operation to have some history in delta table for this TTV lab

In [0]:
%sql
DELETE FROM
  delta_catalog.delta_db.invoices_ttv
WHERE
  customer_id = 1;

In [0]:
%sql
UPDATE
  delta_catalog.delta_db.invoices_ttv
SET
  quantity = 25
WHERE
  customer_id = 2;

In [0]:
%sql
INSERT INTO delta_catalog.delta_db.invoices_ttv
  SELECT
    *
  FROM
    PARQUET.`abfss://sample-files-container@delta0lake0lab0storageac.dfs.core.windows.net/invoices/invoices_101_200.parquet`;

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv;

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_ttv;

#### Time Travel

In [0]:
%sql
-- Latest Version
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv
WHERE
  customer_id IN (1,2);
-- Note :- customer_id 1 is deleted, customer_id 2 is updated Above.

In [0]:
%sql
-- Time Travel
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv VERSION AS OF 0
WHERE
  customer_id IN (1,2);

In [0]:
%sql
--(0th version)
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv TIMESTAMP AS OF '2026-07-16T11:10:03.000+00:00'
WHERE
  customer_id IN (1, 2);

-- Note :- If provided timestamp is not the one given in the history but lies between the timestamp of first and last versions, then it will return the previous version of the table from the given timestamp. else it will throw the error.

-- So below i provided timestamp which is not in the history but lies between the timestamp of first and last versions.
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv TIMESTAMP AS OF '2026-07-16T11:10:05.000+00:00'
WHERE
  customer_id IN (1, 2);

In [0]:
%sql
RESTORE TABLE delta_catalog.delta_db.invoices_ttv TO VERSION AS OF 0;

-- OR alternatively using timestamp
-- RESTORE TABLE delta_catalog.delta_db.invoices_ttv TO TIMESTAMP AS OF '2026-07-16T11:11:15.000+00:00';

In [0]:
%sql
SELECT
  *
FROM
  delta_catalog.delta_db.invoices_ttv
WHERE
  customer_id IN (1,2);

-- Note :- As expected, the result has customer_id = 1 present, customer_id 2 with not update as we have reverted to 0th version.

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_ttv
-- operationMetrics shows the number of files removed and restored and more.
-- operationParameters shows the version restored.

### Note :- We have reverted delta table to 0st version Above. 

###Using Python To Work With Delta Table.

In [0]:
import pyspark.sql.functions as f

In [0]:
# Version 2 will have delete and update changes.
df = spark.read.option("versionAsOf", "2").table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())

# Version 1 will have only the deleted changes.
df = spark.read.option("versionAsOf", "1").table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())

# Latest version is reverted to version 0th in above some cell.
df = spark.read.table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())

In [0]:
%sql
DESCRIBE HISTORY delta_catalog.delta_db.invoices_ttv;

In [0]:
# Timestamp for V2 is 2026-07-16T11:12:15.000+00:00
# Version 2 will have delete and update changes.
df = spark.read.option("timestampAsOf", "2026-07-16T11:12:15.000+00:00").table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())

# Timestamp for V1 is 2026-07-16T11:11:15.000+00:00
# Version 1 will have only the deleted changes.
df = spark.read.option("timestampAsOf", "2026-07-16T11:11:15.000+00:00").table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())

# Setting incorect timestamp btw V3 and V4, V3 will be returned. 
df = spark.read.option("timestampAsOf", "2026-07-16T11:20:22.000+00:00").table("delta_catalog.delta_db.invoices_ttv")
display(df.filter(f.col("customer_id").isin(1,2) ))
print("No. of row",df.count())